<a href="https://colab.research.google.com/github/alk05/routing-comparison-nwm-nextgen-troute/blob/main/restart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Restart Files

In [ ]:
!pip install netCDF4
import xarray as xr
import numpy as np
import json
import pandas as pd
from pathlib import Path
import tempfile
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.6 MB/s eta 0:00:00


In [ ]:
# analysis_assim.channel from Google bucket
nwm_file_path = '/content/drive/MyDrive/Research/nwm.t00z.analysis_assim.channel_rt.tm00.conus.nc'
nwm_ds = xr.open_dataset(nwm_file_path, engine='netcdf4')

# Load routelink
routelink_file_path = '/content/drive/MyDrive/Research/RouteLink_CONUS.nc'
routelink_ds = xr.open_dataset(routelink_file_path, engine='netcdf4')

# Load cat_map_temp
cat_map_file_path = '/content/drive/MyDrive/Research/nwm_to_ngen_map.json'
with open(cat_map_file_path, 'r') as f:
    cat_map_temp = json.load(f)

In [ ]:
# Preprocess cat_map_temp to get NGEN IDs as keys
cat_map_temp_processed = {
    k[4:]: v for k, v in cat_map_temp.items()
}  # remove prefix from keys

# Create crosswalk_ds using the NGEN IDs from cat_map_temp_processed
ngen_link_ids = sorted([int(float(k)) for k in cat_map_temp_processed.keys()])

crosswalk_ds = xr.Dataset(
    coords={
        'link': ngen_link_ids
    }
)

In [ ]:
B2MB = 1048576

def average_nwm_variables(
    nwm_ids_flat: np.ndarray, cat_ids_flat: np.ndarray, nwm_ds: xr.Dataset
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Vectorized averaging calculations for NWM data

    Parameters:
    - nwm_ids_flat: array of NWM ids (np.ndarray)
    - cat_ids_flat: array of NextGen cat-ids (np.ndarray)
    - nwm_ds: NWM analysis/assimilation data (xr.Dataset)

    Returns:
    - nwm_agg: averaged NWM data (pd.DataFrame)
    - mapping_df: DataFrame version of flat maps (pd.DataFrame)
    """

    # --- NWM dataset: streamflow and velocity ---
    # Filter nwm_ds to only feature_ids we care about
    valid_mask = np.isin(nwm_ds["feature_id"].values, nwm_ids_flat)
    nwm_sub = nwm_ds.isel(feature_id=valid_mask)

    # Build a df with feature_id -> cat_id, then merge with nwm values
    mapping_df = pd.DataFrame({"feature_id": nwm_ids_flat, "cat_id": cat_ids_flat})

    nwm_df = pd.DataFrame(
        {
            "feature_id": nwm_sub["feature_id"].values,
            "streamflow": nwm_sub["streamflow"].values,
            "velocity": nwm_sub["velocity"].values,
        }
    )

    nwm_df = nwm_df.merge(mapping_df, on="feature_id", how="left")
    nwm_agg = nwm_df.groupby("cat_id")[["streamflow", "velocity"]].mean()

    return nwm_agg, mapping_df


def average_rtlink_variables(
    nwm_ids_flat: np.ndarray, mapping_df: pd.DataFrame, routelink_ds: xr.Dataset
) -> pd.DataFrame:
    """
    Vectorized averaging calculations for RouteLink data

    Parameters
    - nwm_ids_flat: array of NWM ids (np.ndarray)
    - mapping_df: dataframe of flat map (pd.DataFrame)
    - routelink_ds: NWM RouteLink data (xr.Dataset)

    Returns:
    - rl_agg: averaged NWM RouteLink channel geometry data (pd.DataFrame)
    """

    # --- Routelink dataset: TopWdth, BtmWdth, ChSlp ---
    valid_mask_rl = np.isin(routelink_ds["link"].values, nwm_ids_flat)
    rl_sub = routelink_ds.isel(feature_id=valid_mask_rl)

    rl_df = pd.DataFrame(
        {
            "feature_id": rl_sub["link"].values,
            "TopWdth": rl_sub["TopWdth"].values,
            "BtmWdth": rl_sub["BtmWdth"].values,
            "ChSlp": rl_sub["ChSlp"].values,
        }
    )

    rl_df = rl_df.merge(mapping_df, on="feature_id", how="left")
    rl_agg = rl_df.groupby("cat_id")[["TopWdth", "BtmWdth", "ChSlp"]].mean()

    return rl_agg


def quadratic_formula(b_coeff: np.ndarray, c_coeff: np.ndarray) -> np.ndarray:
    """
    Vectorized quadratic formula solver (assumes no a coefficient). Only returns positive root

    Parameters:
    - b_coeff: np.ndarray
    - c_coeff: np.ndarray

    Returns:
    - h_positive: positive root (np.ndarray)
    """
    discriminant = b_coeff**2 - 4 * c_coeff
    h_positive = (-b_coeff + np.sqrt(discriminant)) / 2

    return h_positive


def solve_depth_geom(
    streamflow: np.ndarray,
    velocity: np.ndarray,
    tw: np.ndarray,
    bw: np.ndarray,
    cs: np.ndarray,
) -> np.ndarray:
    """
    Solves for depth h using CHRTOUT file variables and channel geometry variables.

    Parameters:
    - streamflow: Streamflow from CHRTOUT file. (m^3/s)
    - velocity: Velocity from CHRTOUT file. (m/s)
    - tw: Top width of the main channel. (m)
    - bw: Bottom width of the main channel. (m)
    - cs: Channel slope (dimensionless).

    Returns:
    - h: Initial depth that achieves the target flow rate, or NaN if no solution is found.
    """

    area = np.where(velocity == 0, 0, streamflow / velocity) # cross-sectional area of initial flow
    area = np.where(np.isnan(area), 0, area)  # set NaN areas to 0
    area = np.where(np.isinf(area), 0, area)  # set infinite areas to 0

    db = (cs * (tw - bw)) / 2  # bankfull depth
    area_bankfull = (tw + bw) / 2 * db  # cross-sectional area at bankfull conditions
    # assume trapezoidal main channel

    depths = np.zeros_like(area)  # initialize depths array with 0 values

    above_bankfull = area >= area_bankfull
    area_flood = area[above_bankfull] - area_bankfull[above_bankfull]
    df = area_flood / (tw[above_bankfull] * 3)
    depths[above_bankfull] = db[above_bankfull] + df

    # Below bankfull - solve quadratic formula directly (vectorized)
    below_bankfull = ~above_bankfull & (area > 0)

    # Quadratic: h^2 + cs*bw*h - cs*area = 0
    # Using formula: h = (-b + sqrt(b^2 - 4*c)) / 2, where a=1
    h_positive = quadratic_formula(
        cs[below_bankfull] * bw[below_bankfull],
        -cs[below_bankfull] * area[below_bankfull],
    )
    depths[below_bankfull] = h_positive

    return depths


def create_restart(
    cat_map_temp: dict,
    crosswalk_ds: xr.Dataset,
    nwm_ds: xr.Dataset,
    routelink_ds: xr.Dataset,
) -> xr.Dataset:
    """
    Creates t-route restart file, with 'links' corresponding to NWM IDs.

    Parameters:
    - cat_map_temp: NGEN to NWM catchment json file (dict)
    - crosswalk_ds: "crosswalk" NetCDF file that has all the
    NextGen catchments in the order that the restart file will have them in (xr.Dataset)
    - nwm_ds: NWM analysis/assimilation NetCDF (xr.Dataset)
    - routelink_ds: NWM RouteLink data (xr.Dataset)

    Returns:
    - restart: t-route ingestible restart file (xr.Dataset) with NWM IDs as links
    """
    cat_map_temp_processed = {
        k[4:]: v for k, v in cat_map_temp.items()
    }  # remove prefix from keys


    # Create a base DataFrame from routelink_ds, which contains all NWM links that t-route expects. This ensures all relevant links are in the restart file.
    base_df = pd.DataFrame({
        "feature_id": routelink_ds["link"].values,
        "TopWdth": routelink_ds["TopWdth"].values,
        "BtmWdth": routelink_ds["BtmWdth"].values,
        "ChSlp": routelink_ds["ChSlp"].values,
        "to": routelink_ds["to"].values,
        "from": routelink_ds["from"].values
    }).drop_duplicates(subset=['feature_id']).set_index("feature_id")
    # Sort by index (NWM feature_id) to ensure consistent 'links' order
    base_df = base_df.sort_index()

    # Prepare nwm_ds data
    nwm_data_df = pd.DataFrame({
        "feature_id": nwm_ds["feature_id"].values,
        "streamflow": nwm_ds["streamflow"].values,
        "velocity": nwm_ds["velocity"].values,
    }).drop_duplicates(subset=['feature_id']).set_index("feature_id")

    # Prepare NGEN ID mapping. Map NWM IDs to NGEN IDs.
    nwm_to_ngen_map = {}
    for ngen_id_str, nwm_ids_list in cat_map_temp_processed.items():
        ngen_id = int(float(ngen_id_str)) # Convert NGEN ID string back to int
        for nwm_id_float in nwm_ids_list:
            nwm_to_ngen_map[int(nwm_id_float)] = ngen_id # Store as int

    ngen_mapping_df = pd.DataFrame.from_dict(nwm_to_ngen_map, orient='index', columns=['ngen_id'])
    ngen_mapping_df.index.name = 'feature_id'

    # Merge all data into result_df, using base_df (routelink IDs) as the core. Using left merges ensures that all feature_ids from base_df (routelink_ds) are kept.
    result_df = base_df.merge(nwm_data_df, left_index=True, right_index=True, how='left')
    result_df = result_df.merge(ngen_mapping_df, left_index=True, right_index=True, how='left')

    # Streamflow/velocity/depth for links not in NWM_ds will be 0.
    result_df = result_df.fillna(0)

    # Map 'to' streamflow to 'qlink2'
    streamflow_series = result_df['streamflow']
    result_df['downstream_streamflow'] = result_df['to'].map(streamflow_series).fillna(0)

    # depth calculation
    depths = solve_depth_geom(
        streamflow=np.array(result_df["streamflow"].values),
        velocity=np.array(result_df["velocity"].values),
        tw=np.array(result_df["TopWdth"].values),
        bw=np.array(result_df["BtmWdth"].values),
        cs=np.array(result_df["ChSlp"].values),
    )
    result_df["depth"] = depths

    # create netcdf, with NWM IDs as 'links'
    restart = xr.Dataset(
        data_vars={
            "hlink": (["links"], result_df["depth"].values),
            "qlink1": (["links"], result_df["streamflow"].values),
            "qlink2": (["links"], result_df["downstream_streamflow"].values),
            "ngen_id": (["links"], result_df["ngen_id"].values.astype(int))
        },
        coords={"links": result_df.index.values.astype(str)}, # Cast NWM IDs to string
        attrs={
            "Restart_Time": (
                pd.Timestamp(nwm_ds["time"].values[0]) + pd.Timedelta(hours=1)).strftime(
                "%Y-%m-%d_%H:%M:%S"
            )
        },
    )
    return restart


def write_netcdf_restart(storage_type: str, prefix: Path, ds: xr.Dataset, name: str):
    """
    Write restart data to a NetCDF file.

    Parameters:
        storage_type (str): s3 or local
        prefix (Path): filename prefix
        data (xr.Dataset): restart file
        name (str): string for the filename
    Returns:
        netcdf_cat_file_size (list): file size of output netcdf
    """
    if storage_type == "s3":
        s3_client = boto3.session.Session().client("s3")
        nc_filename = str(prefix) + "/" + name
        bucket, key = convert_url2key(nc_filename, "s3")
        with tempfile.NamedTemporaryFile(suffix=".nc") as tmpfile:
            ds.to_netcdf(tmpfile.name, engine="netcdf4")
            netcdf_cat_file_size = os.path.getsize(tmpfile.name) / B2MB
            tmpfile.flush()
            tmpfile.seek(0)
            print(f"Uploading netcdf forcings to S3: bucket={bucket}, key={key}")
            s3_client.upload_file(tmpfile.name, bucket, key)
    else:
        nc_filename = Path(prefix, name)
        ds.to_netcdf(nc_filename, engine="netcdf4")
        print(f"netcdf has been written to {nc_filename}")
        netcdf_cat_file_size = os.path.getsize(nc_filename) / B2MB
    return [netcdf_cat_file_size]

In [ ]:
# Call the create_restart function
restart_ds = create_restart(
    cat_map_temp=cat_map_temp,
    crosswalk_ds=crosswalk_ds,
    nwm_ds=nwm_ds,
    routelink_ds=routelink_ds
)
display(restart_ds)

/tmp/ipykernel_1762/771905390.py:113: RuntimeWarning: divide by zero encountered in divide
  area = np.where(velocity == 0, 0, streamflow / velocity) # cross-sectional area of initial flow
/tmp/ipykernel_1762/771905390.py:113: RuntimeWarning: invalid value encountered in divide
  area = np.where(velocity == 0, 0, streamflow / velocity) # cross-sectional area of initial flow


<xarray.Dataset> Size: 322MB
Dimensions:  (links: 2776734)
Coordinates:
  * links    (links) <U21 233MB '101' '179' '181' ... '1180001803' '1180001804'
Data variables:
    hlink    (links) float64 22MB 0.3531 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    qlink1   (links) float64 22MB 0.29 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    qlink2   (links) float64 22MB 0.32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    ngen_id  (links) int64 22MB 2440460 2131 2131 2132 2132 2223 ... 0 0 0 0 0 0
Attributes:
    Restart_Time:  2026-04-30_01:00:00

In [ ]:
specific_link_id_int = 22274808
link_data = restart_ds.sel(links=str(specific_link_id_int))

display(link_data)

<xarray.Dataset> Size: 116B
Dimensions:  ()
Coordinates:
    links    <U21 84B '22274808'
Data variables:
    hlink    float64 8B 0.556
    qlink1   float64 8B 2.25
    qlink2   float64 8B 2.53
    ngen_id  int64 8B 497369
Attributes:
    Restart_Time:  2026-04-30_01:00:00

In [ ]:
output_filename = "troute_restart.nc"
output_directory = "/content/"

write_netcdf_restart(
    storage_type="local",
    prefix=output_directory,
    ds=restart_ds,
    name=output_filename
)

netcdf has been written to /content/april30tm01troute_restart.nc


[233.80344581604004]